# R1.4: компактная память и честное возобновление
Проверка хранения, не научная карта. Большой ресурсный опыт не запускается этой тетрадью.

In [ ]:
from pathlib import Path
import os,sys,json,tempfile
root=Path.cwd()
while root!=root.parent and not (root/'efc/model.py').exists(): root=root.parent
assert (root/'efc/model.py').exists()
os.chdir(root);sys.path.insert(0,str(root))
from efc.model import Config,simulate
from efc.records import materialize_result
c=Config(n_cloud=40,n_tasks=500,arrival_rate_s=900,buffer=0)
a=simulate(c);b=simulate(c,compact=True)
assert a==materialize_result(b)
print('Полные записи, входы, очередь, метрики и целые тики совпали.')

In [ ]:
from efc.pilot import _record_run
row=dict(seed=100,warmup=2000,observe=4000,guard=1000,arrivals=8,reused=False)
s={'runs':[]}
_record_run(s,row);_record_run(s,{**row,'reused':True})
assert len(s['runs'])==1 and s['journal_replays']==1
print('Один запрос остаётся одним наблюдением; replay не второй опыт.')

In [ ]:
from scripts.verify_r14_archive import verify_archive,file_hash
import zipfile
with tempfile.TemporaryDirectory() as tmp:
    p=Path(tmp);z=p/'source.zip';d=p/'tree';d.mkdir()
    with zipfile.ZipFile(z,'w') as f:f.writestr('record.json','{}')
    with zipfile.ZipFile(z) as f:f.extractall(d)
    verified=verify_archive(z,d,expected_sha256=file_hash(z))
    assert verified['extracted_members_verified']==1
    (d/'record.json').write_text('changed')
    try: verify_archive(z,d,expected_sha256=file_hash(z))
    except ValueError: pass
    else: raise AssertionError('Повреждение не обнаружено')
out=root/'results/r14/current/storage';out.mkdir(parents=True,exist_ok=True)
(out/'notebook-checks.json').write_text(json.dumps({'full_records_equal':True,'unique_journal_runs':1,'zip_corruption_detected':True,'scientific_grid_run':False},indent=2)+'\n')
print('Повреждённый архив/распаковка не принимаются. Карта не рассчитывалась.')